
# Baseline Model: TF-IDF + Logistic Regression

This notebook builds a baseline for predicting the six Jigsaw toxicity labels. Comments can have multiple labels, so each label is treated as a separate binary classification task.

We reuse the saved 70/15/15 train, validation, and test partitions. TF-IDF is fitted only on training comments, and logistic regression uses balanced class weights to address label imbalance.

Validation performance is measured using average precision for each label, alongside its positive count. The test partition is reserved for final evaluation after model settings and decision thresholds are fixed.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import joblib 

df = pd.read_csv("../data/raw/train.csv")
test_ids = pd.read_csv("../data/processed/test_ids.csv")
train_ids = pd.read_csv("../data/processed/train_ids.csv")
validation_ids = pd.read_csv("../data/processed/validation_ids.csv")
run_id = 'baseline_v1_full_train'

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

In [25]:
test_ids['partition'] = 'test'
train_ids['partition'] = 'train'
validation_ids['partition'] = 'validation'
id_maps = pd.concat([test_ids, train_ids, validation_ids])
id_maps

,id,partition
0,000113f07ec002fd,test
1,00040093b2687caa,test
2,0006f16e4e9f292e,test
3,00078f8ce7eb276d,test
4,0009801bd85e5806,test
...,...,...
23931,ff9ae4a64d2d5de1,validation
23932,ffb47123b2d82762,validation
23933,ffbdbb0483ed0841,validation
23934,ffc2f409658571f1,validation


In [26]:
df = df.merge(
    on = 'id',
    how = 'left',
    validate = '1:1',
    right = id_maps 
)
df

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,partition
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,train
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,train
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,test
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,train
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0,train
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0,train
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0,train
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0,test


In [27]:
missing_partitions = df['partition'].isna()
blank_partitions = df['partition'].fillna('').str.strip().eq('')

print("Missing:", missing_partitions.sum())
print("Blank, excluding missing:", (blank_partitions & ~missing_partitions).sum())
print("Total Count: ", df['partition'].value_counts())

Missing: 0
Blank, excluding missing: 0
Total Count:  partition
train         111699
test           23936
validation     23936
Name: count, dtype: int64


In [28]:
label_sums = df.groupby('partition')[labels].sum()

label_sums = label_sums.loc[['train', 'validation', 'test']].T
label_sums

partition,train,validation,test
toxic,10706,2294,2294
severe_toxic,1116,240,239
obscene,5914,1267,1268
threat,335,71,72
insult,5514,1182,1181
identity_hate,983,211,211


Added a partition column mapped to the source dataset

Validated the results to make sure they match our expected values

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer = 'word',
    ngram_range=(1,1),
    max_features=30000,
    min_df = 2,
    max_df = 1.0,
    lowercase = True,
    stop_words= None,
    sublinear_tf = True,
    norm = 'l2'
)

Created the vectorizer: utilizing TfidVectorizer from sci-kit learn: Comments are represented using word level TF-IDF features. 

In [30]:
train_mask = df['partition'].eq('train')
val_mask = df['partition'].eq('validation')

X_train = vectorizer.fit_transform(
    df.loc[train_mask, 'comment_text']
)

X_val = vectorizer.transform(
    df.loc[val_mask, 'comment_text']
)

print(X_train.shape)
print(X_val.shape)
from scipy.sparse import issparse

assert issparse(X_train) and issparse(X_val)
assert np.isfinite(X_train.data).all()
assert np.isfinite(X_val.data).all()
assert 0 < len(vectorizer.vocabulary_) <= 30000
assert X_train.shape[1] == X_val.shape[1]

(111699, 30000)
(23936, 30000)


Shape is as expected: 111699 rows for train, 23936 for validation. Assertions ran as well

In [31]:
from sklearn.linear_model import LogisticRegression

toxic_model = LogisticRegression(
    class_weight = 'balanced',
    C = 1.0,
    solver = 'liblinear',
    max_iter = 200,
    random_state = 42
)

y_train_toxic = df.loc[train_mask, 'toxic']
toxic_model.fit(X_train, y_train_toxic)

toxic_scores = toxic_model.predict_proba(X_val)
print(toxic_model.classes_)
print(toxic_scores.shape)

[0 1]
(23936, 2)


In [32]:
severe_toxic_model = LogisticRegression(
    class_weight = 'balanced',
    C = 1.0,
    solver = 'liblinear',
    max_iter = 200,
    random_state = 42
)

y_train_severe_toxic = df.loc[train_mask, 'severe_toxic']
severe_toxic_model.fit(X_train, y_train_severe_toxic)

severe_toxic_scores = severe_toxic_model.predict_proba(X_val)
print(severe_toxic_model.classes_)
print(severe_toxic_scores.shape)

[0 1]
(23936, 2)


In [33]:
obscene_model = LogisticRegression(
    class_weight = 'balanced',
    C = 1.0,
    solver = 'liblinear',
    max_iter = 200,
    random_state = 42
)

y_train_obscene = df.loc[train_mask, 'obscene']
obscene_model.fit(X_train, y_train_obscene)

obscene_scores = obscene_model.predict_proba(X_val)
print(obscene_model.classes_)
print(obscene_scores.shape)

[0 1]
(23936, 2)


In [34]:
threat_model = LogisticRegression(
    class_weight = 'balanced',
    C = 1.0,
    solver = 'liblinear',
    max_iter = 200,
    random_state = 42
)

y_train_threat = df.loc[train_mask, 'threat']
threat_model.fit(X_train, y_train_threat)

threat_scores = threat_model.predict_proba(X_val)
print(threat_model.classes_)
print(threat_scores.shape)

[0 1]
(23936, 2)


In [35]:
insult_model = LogisticRegression(
    class_weight = 'balanced',
    C = 1.0,
    solver = 'liblinear',
    max_iter = 200,
    random_state = 42
)

y_train_insult = df.loc[train_mask, 'insult']
insult_model.fit(X_train, y_train_insult)

insult_scores = insult_model.predict_proba(X_val)
print(insult_model.classes_)
print(insult_scores.shape)

[0 1]
(23936, 2)


In [36]:
identity_hate_model = LogisticRegression(
    class_weight = 'balanced',
    C = 1.0,
    solver = 'liblinear',
    max_iter = 200,
    random_state = 42
)

y_train_identity_hate = df.loc[train_mask, 'identity_hate']
identity_hate_model.fit(X_train, y_train_identity_hate)

identity_hate_scores = identity_hate_model.predict_proba(X_val)
print(identity_hate_model.classes_)
print(identity_hate_scores.shape)

[0 1]
(23936, 2)


Trained all 6 models for all classifier tags